# Fraud Detection — Exploratory Data Analysis
## IEEE-CIS Financial Transaction Dataset

**Business Context:**  
Card fraud costs the global financial industry $32B+ annually. This project builds a production-grade fraud detection system on 590,540 real-world transactions from the IEEE-CIS Kaggle competition dataset — replicating the exact problem Amex, Citi, and HSBC solve at scale.

**Objective:** Identify fraudulent transactions with high recall (minimize missed fraud) while controlling false positive rate (avoid blocking legitimate customers).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_palette('husl')

print('Libraries loaded successfully')

## 1. Data Loading
IEEE-CIS dataset: 2 files — `train_transaction.csv` and `train_identity.csv`  
Download from: https://www.kaggle.com/competitions/ieee-fraud-detection/data

In [ ]:
# Load transaction data (590,540 rows x 394 features)
df_trans = pd.read_csv('../data/raw/train_transaction.csv')
df_id    = pd.read_csv('../data/raw/train_identity.csv')

# Merge on TransactionID
df = df_trans.merge(df_id, on='TransactionID', how='left')

print(f'Total transactions : {len(df):,}')
print(f'Total features     : {df.shape[1]}')
print(f'Fraud cases        : {df["isFraud"].sum():,} ({df["isFraud"].mean()*100:.2f}%)')
print(f'Legit cases        : {(df["isFraud"]==0).sum():,}')
df.head(3)

## 2. Class Imbalance — The Core Challenge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Raw)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Transaction Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold')

# Log scale
axes[1].bar(['Legitimate', 'Fraud'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1].set_yscale('log')
axes[1].set_title('Class Distribution (Log Scale)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Transaction Count (log)')

plt.suptitle('IEEE-CIS Dataset: Severe Class Imbalance (~3.5% Fraud Rate)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/01_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBusiness Insight: Imbalanced data means accuracy alone is misleading.')
print('A model predicting ALL transactions as legitimate gets 96.5% accuracy but catches ZERO fraud.')
print('We optimize for RECALL (catch fraud) while controlling false positive rate.')

## 3. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by fraud label
for label, color, name in [(0,'#2ecc71','Legitimate'), (1,'#e74c3c','Fraud')]:
    subset = df[df['isFraud']==label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=50, alpha=0.6,
                 color=color, label=f'{name} (n={len(subset):,})', density=True)
axes[0].set_xlabel('log(Transaction Amount + 1)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount Distribution by Fraud Label')
axes[0].legend()

# Box plot
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[1],
           patch_artist=True)
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
axes[1].set_title('Amount Spread by Fraud Label')
axes[1].set_xlabel('')
axes[1].set_ylabel('Transaction Amount ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../outputs/plots/02_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Legit  | Mean: ${df[df["isFraud"]==0]["TransactionAmt"].mean():.2f} | Median: ${df[df["isFraud"]==0]["TransactionAmt"].median():.2f}')
print(f'Fraud  | Mean: ${df[df["isFraud"]==1]["TransactionAmt"].mean():.2f} | Median: ${df[df["isFraud"]==1]["TransactionAmt"].median():.2f}')

## 4. Temporal Patterns — When Does Fraud Happen?

In [ ]:
# TransactionDT is seconds from reference point
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  = (df['TransactionDT'] // (3600 * 24)) % 7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by hour
hourly = df.groupby('hour')['isFraud'].agg(['sum','count'])
hourly['fraud_rate'] = hourly['sum'] / hourly['count'] * 100
axes[0].plot(hourly.index, hourly['fraud_rate'],
             color='#e74c3c', linewidth=2.5, marker='o', markersize=4)
axes[0].fill_between(hourly.index, hourly['fraud_rate'], alpha=0.15, color='#e74c3c')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Hour of Day')
axes[0].axhline(df['isFraud'].mean()*100, color='gray',
                linestyle='--', label='Overall avg')
axes[0].legend()

# Fraud rate by day
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily = df.groupby('day')['isFraud'].agg(['sum','count'])
daily['fraud_rate'] = daily['sum'] / daily['count'] * 100
axes[1].bar(range(len(daily)), daily['fraud_rate'],
            color=['#e74c3c' if r == daily['fraud_rate'].max() else '#3498db'
                   for r in daily['fraud_rate']])
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(days)
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Day of Week')

plt.tight_layout()
plt.savefig('../outputs/plots/03_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Product Code & Card Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by ProductCD
prod = df.groupby('ProductCD')['isFraud'].agg(['mean','count']).reset_index()
prod['mean'] *= 100
bars = axes[0].bar(prod['ProductCD'], prod['mean'],
                   color=['#e74c3c' if x == prod['mean'].max() else '#3498db'
                          for x in prod['mean']])
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Product Code')
for bar, (_, row) in zip(bars, prod.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.1,
                 f"{row['mean']:.1f}%", ha='center', fontsize=9)

# Fraud rate by card type
if 'card4' in df.columns:
    card = df.groupby('card4')['isFraud'].agg(['mean','count']).reset_index()
    card['mean'] *= 100
    card = card.dropna()
    axes[1].bar(card['card4'], card['mean'], color='#9b59b6')
    axes[1].set_ylabel('Fraud Rate (%)')
    axes[1].set_title('Fraud Rate by Card Network')

plt.tight_layout()
plt.savefig('../outputs/plots/04_product_card_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Missing Value Heatmap

In [ ]:
# Missing value analysis — key columns only
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_top = missing[missing > 0].head(40)

plt.figure(figsize=(12, 8))
colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#2ecc71'
          for x in missing_top.values]
bars = plt.barh(missing_top.index, missing_top.values, color=colors)
plt.axvline(50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
plt.axvline(20, color='orange', linestyle='--', alpha=0.5, label='20% threshold')
plt.xlabel('Missing Value %')
plt.title('Top 40 Features by Missing Rate\n(Red >50% | Orange >20% | Green <20%)',
          fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/plots/05_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Features with >50% missing: {(missing > 50).sum()}')
print(f'Features with >20% missing: {(missing > 20).sum()}')
print(f'Features with 0% missing  : {(missing == 0).sum()}')

## 7. EDA Summary — Key Business Findings

| Finding | Business Implication |
|---------|---------------------|
| 3.5% fraud rate → severe imbalance | Must use SMOTE + recall-focused metrics |
| Fraud transactions have higher avg amount | Amount is a strong fraud signal |
| Off-peak hours show elevated fraud rate | Time-based rules can filter suspicious activity |
| Certain ProductCDs have 2–3× higher fraud rate | Product-level risk scoring needed |
| 100+ features with >50% missing | Aggressive imputation + feature selection required |